# SMC Course - Tagalog transcription from YouTube (free GPU)

**Steps:**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Run cell 1 and 2 (GPU check + install).
3. In **cell 3**, paste each YouTube link in the quotes next to its lesson.
4. Run cell 4 (transcribe) and cell 5 (download the zip).

Only lessons that have a real link (not 'PASTE_URL_HERE') will be transcribed, so you can do them in batches.

In [ ]:
!nvidia-smi

In [ ]:
!pip -q install -U openai-whisper yt-dlp

In [ ]:
# Paste the YouTube link for each lesson inside the quotes.
VIDEOS = {
  'Part 1/Part 1 Lesson 1 - States of the Market':        'PASTE_URL_HERE',
  'Part 1/Part 1 Lesson 2 - OHLC & OLHC':                 'PASTE_URL_HERE',
  'Part 1/Part 1 Lesson 3 - Liquidity Pools':             'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 1 - Order Flow':                  'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 2 - Generated Liquidity':         'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 3 - ERL & IRL':                   'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 4 - Market Structure':            'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 5 - SMT Divergence':              'PASTE_URL_HERE',
  'Part 2/Part 2 Lesson 6 - Standard Deviation Projection':'PASTE_URL_HERE',
  'Part 3/Part 3 Lesson 1 - MMXMs':                       'PASTE_URL_HERE',
  'Part 3/Part 3 Lesson 2 - Catching Expansions':         'PASTE_URL_HERE',
  'Part 3/Part 3 Lesson 3 - Entry Patterns':              'PASTE_URL_HERE',
  'Part 3/Part 3 Lesson 4 - Combining Everything':        'PASTE_URL_HERE',
  'Part 4/Part 4 Lesson 1 - Asian Session High & Low':    'PASTE_URL_HERE',
  'Part 4/Part 4 Lesson 2 - Main Model':                  'PASTE_URL_HERE',
  'Part 4/Part 4 Lesson 3 - Previous Day High & Low Model':'PASTE_URL_HERE',
}
MODEL = 'large-v3'   # most accurate on GPU; use 'medium' to go faster
LANG  = 'tl'         # Tagalog

In [ ]:
import os, glob, subprocess, tempfile, whisper
OUT = '/content/transcripts'
todo = [(n, u) for n, u in VIDEOS.items() if u and 'PASTE_URL' not in u]
print(f'{len(todo)} videos to transcribe')
model = whisper.load_model(MODEL)
for i, (name, url) in enumerate(todo, 1):
    out = os.path.join(OUT, name + '.txt')
    os.makedirs(os.path.dirname(out), exist_ok=True)
    print(f'[{i}/{len(todo)}] {name}')
    with tempfile.TemporaryDirectory() as td:
        subprocess.run(['yt-dlp', '-f', 'bestaudio/best', '-x', '--audio-format', 'mp3',
                        '-o', f'{td}/a.%(ext)s', url], check=True)
        audio = glob.glob(f'{td}/a.*')[0]
        r = model.transcribe(audio, language=LANG, fp16=True, verbose=False)
    with open(out, 'w', encoding='utf-8') as f:
        f.write(r['text'].strip() + '\n')
    print('    words:', len(r['text'].split()))
print('Done ->', OUT)

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('/content/transcripts', 'zip', '/content/transcripts')
files.download('/content/transcripts.zip')